In [2]:
import json
import pandas as pd

def safe_get_first(data_list):
    """Lấy phần tử đầu tiên của danh sách một cách an toàn."""
    if isinstance(data_list, list) and len(data_list) > 0:
        return data_list[0]
    return {}

def extract_and_save_creators(input_file, output_csv):
    # Dictionary để lưu trữ thông tin gộp của từng creator
    # Key là aioCreatorID
    creators_master = {}

    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            try:
                line_data = json.loads(line)
            except: continue
            
            for req in line_data.get('requests', []):
                body_raw_str = req.get('body_raw')
                if not body_raw_str: continue
                
                try:
                    body_data = json.loads(body_raw_str)
                    creators = body_data.get('creators', [])
                except: continue
                
                for c in creators:
                    cid = c.get('aioCreatorID')
                    if not cid: continue
                    
                    # Nếu chưa có creator này, khởi tạo bản ghi mới
                    if cid not in creators_master:
                        creators_master[cid] = {
                            "info": {},
                            "videos": {} # Dùng dict để deduplicate video theo itemID
                        }
                    
                    master = creators_master[cid]
                    tt_info = c.get('creatorTTInfo', {})
                    es_data = c.get('esData', {})
                    price = es_data.get('price', {})
                    cs = c.get('creditScore', {})
                    ri = c.get('riskInfo', {})
                    cvs = c.get('creatorValueStat', {})
                    stat_data = c.get('statisticData', {})
                    op = stat_data.get('overallPerformance', {})

                    fdd = stat_data.get('followerDistriData', {})
                    
                    # Gộp thông tin (chỉ ghi đè nếu dữ liệu mới không rỗng)
                    def update_if_exists(target, source_dict, mapping):
                        for key, path in mapping.items():
                            val = source_dict.get(path)

                            if val is None or val == "":
                                continue

                            # 🔥 chỉ ghi nếu chưa có
                            if key not in target:
                                target[key] = val
                    if isinstance(op, list):
                        op = safe_get_first(op)


                    update_if_exists(master['info'], op, {
                        "avgSixSecondsViewsBenchMarkViews": "avgSixSecondsViewsBenchMarkViews",
                        "avgSixSecondsViewsRate": "avgSixSecondsViewsRate",
                        "avgSixSecondsViewsRateRank": "avgSixSecondsViewsRateRank",
                        "engagementRateBenchMark": "engagementRateBenchMark",
                        "engagementRateRank": "engagementRateRank",
                        "followerTier": "followerTier",
                        "followersGrowthRate": "followersGrowthRate",
                        "followersGrowthRateRank": "followersGrowthRateRank",
                        "medianBenchMarkViews": "medianBenchMarkViews",
                        "medianViewsRank": "medianViewsRank",
                        "videoCompleteRate": "videoCompleteRate",
                        "videoCompleteRateRank": "videoCompleteRateRank"
                    })
                    update_if_exists(master['info'], tt_info, {
                        "tiktokUID": "ttUID", "nickname": "nickName", 
                        "handleName": "handleName", "followerCnt": "followerCnt"
                    })
                    update_if_exists(master['info'], c, {"displayType": "displayType"})
                    update_if_exists(master['info'], price, {
                        "recommendRate100k": "recommendRate100k", "startingRate100k": "startingRate100k"
                    })
                    update_if_exists(master['info'], cs, {
                        "currentScore": "currentScore", "currentTier": "currentTier",
                        "scoreLowerLimit": "scoreLowerLimit", "scoreUpperLimit": "scoreUpperLimit"
                    })
                    update_if_exists(master['info'], op, {"engagementRate": "engagementRate",
                                                           "medianViews": "medianViews"})
                    update_if_exists(master['info'], cvs, {
                        "broadcastingScore": "broadcastingScore", "collaborationScore": "collaborationScore",
                        "commercialScore": "commercialScore", "comprehensiveScore": "comprehensiveScore"
                    })
                    
                    # Xử lý các trường danh sách
                    if c.get('contentLabels'):
                        master['info']['contentLabels'] = "|".join([l.get('labelName', '') for l in c.get('contentLabels', [])])
                    
                    if ri.get('disciplineInfoList'):
                        d = safe_get_first(ri['disciplineInfoList'])
                        master['info']['disciplineStatus'] = d.get('disciplineStatus')
                        master['info']['disciplineType'] = d.get('disciplineType')
                    
                    if ri.get('riskEventInfoList'):
                        r = safe_get_first(ri['riskEventInfoList'])
                        master['info']['riskEventStatus'] = r.get('riskEventStatus')
                        master['info']['riskEventType'] = r.get('riskEventType')

                    # Lưu các thống kê dưới dạng chuỗi
                    for stat_key in ['active', 'age', 'deviceBrand', 'gender', 'region']:
                        if fdd.get(stat_key):
                            master['info'][f"{stat_key}_dist"] = str(fdd[stat_key])

                    # Gom danh sách video
                    for v in c.get('recentItems', []):
                        vid = v.get('itemID')
                        if vid:
                            master['videos'][vid] = {
                                "video_itemID": vid,
                                "video_title": v.get('title'),
                                "video_views": v.get('views'),
                                "video_heart": v.get('heart'),
                                "video_comment": v.get('comment'),
                                "video_share": v.get('share'),
                                "video_createTime": v.get('createTime')
                            }

    # Chuyển đổi dữ liệu đã gộp sang dạng bảng để lưu CSV
    final_rows = []
    for cid, data in creators_master.items():
        base = {"aioCreatorID": cid}
        base.update(data['info'])
        
        if data['videos']:
            for vid, v_info in data['videos'].items():
                row = base.copy()
                row.update(v_info)
                final_rows.append(row)
        else:
            final_rows.append(base)

    df = pd.DataFrame(final_rows)
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"Hoàn thành! Đã xử lý {len(creators_master)} creator và lưu {len(df)} dòng video.")


In [3]:

extract_and_save_creators('results_success.jsonl', 'dataset_creators.csv')

Hoàn thành! Đã xử lý 9935 creator và lưu 144249 dòng video.
